In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
from utils import load_data, evaluate_predictions, TARGET_COLUMNS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TransformerToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True, max_length=max_len, return_tensors="pt"
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item

# 1. Load Data
train_df, val_df = load_data("train.csv")
train_ds = TransformerToxicDataset(train_df["cleaned_text"].values, train_df[TARGET_COLUMNS].values, tokenizer)
val_ds = TransformerToxicDataset(val_df["cleaned_text"].values, val_df[TARGET_COLUMNS].values, tokenizer)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

# 2. Initialize Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(TARGET_COLUMNS),
    problem_type="multi_label_classification"
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# 3. Fine-Tuning Loop
print(f"Fine-tuning {MODEL_NAME}...")
model.train()
for epoch in range(2):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/2 - Loss: {total_loss / len(train_loader):.4f}")

# 4. Evaluation
model.eval()
prob_batches = []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()
        prob_batches.append(probs)

y_probs = np.vstack(prob_batches)
evaluate_predictions("DistilBERT (Fine-Tuned)", val_df[TARGET_COLUMNS].values.astype(np.float32), y_probs)